In [ ]:
<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Three-Pass-Approach/MNPS_Job_Classification_GPT4o_Two_Pass_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# **MNPS_Job_Classification_GPT4o_Updated (Two-Pass + Self-Consistency Fix)**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.5 Changes**
> - **Fixed Two-Pass Logic**: Post-processing now occurs *after* Pass 2 to preserve reasoning integrity
> - **Stronger Self-Consistency Prompt**: Explicit instruction to align classification with justification
> - **Removed `new_job_title` from Pass 2 context**: Prevents role anchoring bias
> - **Added "Liaison" to approved roles**: Reflects MNPS ground truth usage
> - **Preserves all v7.5.4 infrastructure**: Drive mounting, validation, output format, model, etc.
> - **Targets Justification Mismatch Problem**: Rows 16, 26, 29, 36, 38 should now self-correct

In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files — check if in root or inside zip
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

In [ ]:
# ==== 2) Load data and build attribute-only view (ignore title) ====
preds = df.copy()

ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

In [ ]:
# ==== 3) Enhanced closed sets and normalization helpers ====
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# Updated to include "Liaison"
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant', 'Liaison'  # Added per ground truth
]
MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

SPECIALIST_FALLBACKS = [
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),
    ('Analyst', 'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),
    ('Architect (Facility-Focused)', 'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', 'system|software|technology|IT|database|network|programming|technical architecture')
]

EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

def normalize_minor(x: str) -> str:
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    if minor_role == 'Lead':
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    return minor_role

print("✅ Enhanced closed sets and normalization helpers defined (with Liaison)")

In [ ]:
# ==== 4) Build comprehensive KSACs text ====
def build_ksacs_text():
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)
    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"

    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"

    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")

In [ ]:
# ==== 5) Enhanced Zero Shot Prompt ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.
Process:
- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.
- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.
- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.
- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.
- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.
- Never justify classifications based on job titles - only use job attributes and MNPS standards.
IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet):
ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain, but prefer more specific roles when possible
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting
- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management
- **Supervisor vs Manager**: 
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree
- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming
MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity
Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level (consider executive role guidelines)
- new_job_title: Should incorporate both major_role_group and minor_sub_group (e.g., "Accountant II", "Facility Coordinator II")
- Provide detailed justification based on job attributes and MNPS KSACs alignment that matches your selected role and level"""

print("✅ Enhanced zero shot prompt defined")

# NEW: STRONGER Self-Consistency Prompt for Pass 2
self_consistency_prompt = \
"""You are performing a STRICT self-consistency audit of your own prior output.

**RULES:**
1. Read ONLY your `grouping_justification`.
2. Determine what role is **explicitly described** in that justification.
3. Compare it to your stated `major_role_group`.
4. IF THEY DIFFER → **you must change `major_role_group`, `minor_sub_group`, and `new_job_title` to match the justification.**
5. IF THEY MATCH → return unchanged.

**You are NOT allowed to:**
- Keep a classification that contradicts your own reasoning
- Assume the initial prediction was correct
- Modify the justification

**Output must be valid JSON with the exact fields.**"""

print("✅ Stronger self-consistency prompt for Pass 2 defined")

In [ ]:
# ==== 6) OpenAI API Setup ====
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    if model is None:
        model = MODEL_ID
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            error_str = str(e).lower()
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                print(f"❌ Non-rate limiting error: {e}")
                raise e
    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined")

In [ ]:
# ==== 7) Two-Pass Batch Processing with POST-PROCESSING AFTER Pass 2 ====
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # === PASS 1: Initial Classification (raw LLM output) ===
    pass1_prompt = f"""{zero_shot_prompt}
Available MNPS Roles: {', '.join(VALID_ROLES)}
{KSACS_TEXT}
Job Description to Classify:
{job_text}
**IMPORTANT**: 
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
- Apply Problem Role Cheat Sheet guidelines
- Avoid overusing "Specialist"
Return your response as a JSON object with:
{{
  "new_job_title": "...",
  "major_role_group": "...",
  "minor_sub_group": "...",
  "grouping_justification": "..."
}}"""
    
    try:
        # Get initial prediction
        pass1_response = call_llm_json_with_retry(pass1_prompt, MODEL_ID)
        
        # Store raw LLM output (no post-processing yet)
        raw_major = pass1_response.get('major_role_group', 'Other')
        raw_minor = pass1_response.get('minor_sub_group', 'I')
        justification = pass1_response.get('grouping_justification', 'No justification provided')
        raw_title = pass1_response.get('new_job_title', f"{raw_major} {raw_minor}")

        # === PASS 2: Self-Consistency Check — ONLY show justification + classification fields (no title bias) ===
        # Construct minimal prior output (exclude new_job_title to avoid anchoring)
        prior_output = {
            "major_role_group": raw_major,
            "minor_sub_group": raw_minor,
            "grouping_justification": justification
        }
        pass2_prompt = f"""{self_consistency_prompt}

**Your Previous Output:**
{json.dumps(prior_output, indent=2)}

**Now perform the self-consistency check and return the corrected (or unchanged) JSON. Include a reconstructed `new_job_title` that matches the corrected role and level.**
"""
        pass2_response = call_llm_json_with_retry(pass2_prompt, MODEL_ID)

        # Extract Pass 2 output
        major_role = pass2_response.get('major_role_group', raw_major)
        minor_role = pass2_response.get('minor_sub_group', raw_minor)
        new_job_title = pass2_response.get('new_job_title', f"{major_role} {minor_role}")
        # Use justification from Pass 2 (should be unchanged, but safe to take)
        justification = pass2_response.get('grouping_justification', justification)

        # === APPLY RULE-BASED POST-PROCESSING NOW (after Pass 2) ===
        major_role = discourage_specialist(job_text, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)
        minor_role = normalize_minor(minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        if not new_job_title or new_job_title == 'Unknown':
            new_job_title = f"{major_role} {minor_role}"

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': new_job_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'grouping_justification': justification,
            'model_used': MODEL_ID
        }

    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions
results = []
print("🚀 Starting two-pass batch processing with self-consistency fix...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.2)

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v755_two_pass_fixed.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

In [ ]:
# ==== 8) Generate Summary Statistics and Examples ====
preds = results_df.copy()

major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count', 'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})
summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v755_two_pass_fixed.csv"
summary_stats.to_csv(summary_path, index=False)

examples = preds[['source_row_index', 'job_title_original', 'new_job_title', 
                  'major_role_group', 'minor_sub_group']].head(10)
examples_path = OUTPUTS_DIR / "examples_gpt4o_v755_two_pass_fixed.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))
print("\n📝 Major Role Distribution:")
print(major_counts.to_string())
print("\n📝 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n📝 Example Classifications:")
print(examples.to_string(index=False))
print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")

In [ ]:
# ==== 9) Enhanced Quality Check and Validation ====
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })

executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / "alignment_issues_gpt4o_v755_two_pass_fixed.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")

if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / "title_format_issues_gpt4o_v755_two_pass_fixed.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")

if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / "executive_lead_issues_gpt4o_v755_two_pass_fixed.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")

print("\n✅ Enhanced quality check completed")